# M2.2: Recommendation-Aware MiniLM Fine-Tuning

This notebook performs one controlled fine-tuning experiment on the metadata encoder.

**Goal:** adapt `sentence-transformers/all-MiniLM-L6-v2` from generic semantic similarity toward recommendation-aware item similarity using **TRAIN interactions only**.

The experiment does **not** change BPR or the accepted final M4 settings.

Fine-tuning signal:

`adjacent TRAIN co-interactions from the same user -> positive metadata-text pairs`

Training objective:

`MultipleNegativesRankingLoss` with a no-duplicates batch sampler. Other positive items in the batch act as in-batch negatives.

Evaluation is strictly apples-to-apples:

- candidate K = 500 per retriever
- recent interactions = 2
- recency weight `rho = 0.60`
- final reranking = `0.50 * BPR_z + 0.50 * Metadata_z`
- frozen MiniLM and fine-tuned MiniLM are compared on the same VALID users

**TEST is not evaluated here.** It is loaded only to preserve the exact original BPR item mapping.

Saved artifacts are intentionally limited to the expensive outputs:

- `minilm_recommender_finetuned/` - fine-tuned encoder
- `metadata_minilm_finetuned_embeddings.npy` - full metadata embedding matrix

The existing `metadata_embedding_index.parquet` is reused because item order does not change.

## 1. Imports and fixed configuration

In [ ]:
from pathlib import Path
import random
import shutil

import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm

try:
    from datasets import Dataset
    from sentence_transformers import (
        SentenceTransformer,
        SentenceTransformerTrainer,
        SentenceTransformerTrainingArguments,
        losses,
    )
    from sentence_transformers.sentence_transformer.training_args import BatchSamplers
except ImportError as exc:
    raise ImportError(
        "This notebook needs `sentence-transformers` and `datasets`. "
        "Install them with: pip install -U sentence-transformers datasets"
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path('.')

TRAIN_PATH = DATA_DIR / 'train_interactions.parquet'
VALID_PATH = DATA_DIR / 'valid_interactions.parquet'
TEST_PATH = DATA_DIR / 'test_interactions.parquet'
METADATA_PATH = DATA_DIR / 'product_metadata.parquet'

BPR_CHECKPOINT = DATA_DIR / 'bpr_m1a_all_interactions.pt'
FROZEN_EMBED_PATH = DATA_DIR / 'metadata_minilm_embeddings.npy'
M2_INDEX_PATH = DATA_DIR / 'metadata_embedding_index.parquet'

BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
FINETUNED_MODEL_DIR = DATA_DIR / 'minilm_recommender_finetuned'
FINETUNED_EMBED_PATH = DATA_DIR / 'metadata_minilm_finetuned_embeddings.npy'

# Fine-tuning settings. One epoch is deliberate: this is domain adaptation, not training from scratch.
MAX_TRAIN_PAIRS = 60_000
TRAIN_BATCH_SIZE = 64
NUM_EPOCHS = 1
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
MAX_SEQ_LENGTH = 256
ENCODE_BATCH_SIZE = 128

# Frozen downstream settings from the accepted M4 validation pipeline.
CANDIDATE_K = 500
RECENT_N = 2
RHO = 0.60
ALPHA_BPR = 0.50
ALPHA_META = 0.50
EVAL_BATCH_SIZE = 96

print('Device available:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Fine-tuning pairs :', MAX_TRAIN_PAIRS)
print('Train batch size  :', TRAIN_BATCH_SIZE)
print('Epochs            :', NUM_EPOCHS)
print('Learning rate     :', LEARNING_RATE)

## 2. Load interactions and metadata

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
valid = pd.read_parquet(VALID_PATH)
test = pd.read_parquet(TEST_PATH)
metadata = pd.read_parquet(METADATA_PATH, columns=['parent_asin', 'title', 'categories', 'features', 'description']).reset_index(drop=True)
metadata_index = pd.read_parquet(M2_INDEX_PATH)

print('Train   :', train.shape)
print('Valid   :', valid.shape)
print('Test    :', test.shape)
print('Metadata:', metadata.shape)

assert len(metadata) == len(metadata_index)
assert np.array_equal(
    metadata['parent_asin'].astype(str).to_numpy(),
    metadata_index['parent_asin'].astype(str).to_numpy(),
), 'Metadata row order differs from the saved M2 embedding index.'

## 3. Rebuild the exact M2 metadata text

This preserves the final M2 text construction: **title + categories + features + description**. Rating statistics are not included.

In [ ]:
def clean_field(value):
    if value is None:
        return ''

    if isinstance(value, (list, tuple, np.ndarray)):
        return ' '.join(str(x) for x in value if x is not None)

    try:
        if pd.isna(value):
            return ''
    except Exception:
        pass

    return str(value)


metadata = metadata.copy()
for col in ['title', 'categories', 'features', 'description']:
    metadata[col + '_text'] = metadata[col].apply(clean_field)

metadata['content_text'] = (
    'Title: ' + metadata['title_text']
    + '. Categories: ' + metadata['categories_text']
    + '. Features: ' + metadata['features_text']
    + '. Description: ' + metadata['description_text']
)

assert metadata['content_text'].notna().all()
assert (metadata['content_text'].str.len() > 0).all()

print('Metadata texts:', len(metadata))
print('Median chars  :', int(metadata['content_text'].str.len().median()))
print(metadata[['parent_asin', 'content_text']].head(2).to_string(index=False))

## 4. Build recommendation-aware positive pairs from TRAIN only

To avoid letting very active users dominate training, this notebook selects **at most one adjacent chronological item pair per user**.

For a user history such as:

`A -> B -> C -> D`

one transition is sampled reproducibly from `(A,B)`, `(B,C)`, `(C,D)`.

Then a global reproducible sample of at most `MAX_TRAIN_PAIRS` users is used for fine-tuning. Exact self-pairs are removed.

In [ ]:
train_pairs_df = train[['user_id', 'parent_asin', 'timestamp']].copy()
train_pairs_df = train_pairs_df.sort_values(
    ['user_id', 'timestamp'],
    kind='mergesort',
)

rng = np.random.default_rng(SEED)
pair_rows = []

for user_id, group in tqdm(
    train_pairs_df.groupby('user_id', sort=False),
    desc='Build one positive pair/user',
):
    items = group['parent_asin'].to_numpy()
    if len(items) < 2:
        continue

    valid_positions = np.where(items[:-1] != items[1:])[0]
    if len(valid_positions) == 0:
        continue

    j = int(rng.choice(valid_positions))
    pair_rows.append((items[j], items[j + 1]))

positive_pairs = pd.DataFrame(pair_rows, columns=['anchor_asin', 'positive_asin'])
positive_pairs = positive_pairs.drop_duplicates().reset_index(drop=True)

if len(positive_pairs) > MAX_TRAIN_PAIRS:
    positive_pairs = positive_pairs.sample(
        n=MAX_TRAIN_PAIRS,
        random_state=SEED,
    ).reset_index(drop=True)

text_lookup = metadata.set_index('parent_asin')['content_text'].to_dict()

positive_pairs['anchor'] = positive_pairs['anchor_asin'].map(text_lookup)
positive_pairs['positive'] = positive_pairs['positive_asin'].map(text_lookup)
positive_pairs = positive_pairs.dropna(subset=['anchor', 'positive']).reset_index(drop=True)

# Randomly swap direction for half of the pairs so neither chronological side
# is always treated as the anchor.
swap_rng = np.random.default_rng(SEED + 1)
swap_mask = swap_rng.random(len(positive_pairs)) < 0.5
anchor_copy = positive_pairs.loc[swap_mask, 'anchor'].copy()
positive_pairs.loc[swap_mask, 'anchor'] = positive_pairs.loc[swap_mask, 'positive'].to_numpy()
positive_pairs.loc[swap_mask, 'positive'] = anchor_copy.to_numpy()

print('Fine-tuning positive pairs:', len(positive_pairs))
print('Unique anchor items       :', positive_pairs['anchor_asin'].nunique())
print('Unique positive items     :', positive_pairs['positive_asin'].nunique())

assert len(positive_pairs) > 10_000, 'Unexpectedly few training pairs.'

## 5. Create the contrastive training dataset

`MultipleNegativesRankingLoss` treats the matched pair as positive and the other positive texts in the batch as negatives. The no-duplicates batch sampler prevents identical texts/items from appearing twice in one batch, which is especially important for in-batch-negative training.

In [ ]:
train_dataset = Dataset.from_dict({
    'anchor': positive_pairs['anchor'].tolist(),
    'positive': positive_pairs['positive'].tolist(),
})

print(train_dataset)
print(train_dataset[0].keys())

## 6. Fine-tune MiniLM

This is intentionally conservative:

- one epoch
- low learning rate
- no intermediate checkpoint saving
- no VALID or TEST interactions used as supervision

If you are on CPU, this can take noticeably longer than the earlier matrix-based stages. Do not interrupt unless training is clearly stuck.

In [ ]:
encoder = SentenceTransformer(BASE_MODEL)
encoder.max_seq_length = MAX_SEQ_LENGTH

train_loss = losses.MultipleNegativesRankingLoss(encoder)

training_args = SentenceTransformerTrainingArguments(
    output_dir='minilm_finetune_tmp',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy='no',
    eval_strategy='no',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    bf16=False,
    seed=SEED,
)

trainer = SentenceTransformerTrainer(
    model=encoder,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss,
)

trainer.train()
print('Fine-tuning complete.')

## 7. Save only the expensive fine-tuned encoder

The temporary trainer output directory is removed. The final SentenceTransformer directory is retained because reproducing the fine-tuning is expensive and the encoder is needed to embed future/new products.

In [ ]:
if FINETUNED_MODEL_DIR.exists():
    shutil.rmtree(FINETUNED_MODEL_DIR)

encoder.save(str(FINETUNED_MODEL_DIR))

# Remove temporary Trainer output if it exists.
tmp_dir = Path('minilm_finetune_tmp')
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)

print('Saved fine-tuned encoder:', FINETUNED_MODEL_DIR)

## 8. Encode all metadata with the fine-tuned MiniLM

In [ ]:
finetuned_embeddings = encoder.encode(
    metadata['content_text'].tolist(),
    batch_size=ENCODE_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)

assert finetuned_embeddings.shape == (len(metadata), 384)
assert np.isfinite(finetuned_embeddings).all()

np.save(FINETUNED_EMBED_PATH, finetuned_embeddings)

print('Fine-tuned embeddings:', finetuned_embeddings.shape)
print('Saved:', FINETUNED_EMBED_PATH)

## 9. Quick embedding-space sanity check

This is not the recommendation metric. It only verifies that fine-tuning actually changed the representation rather than reproducing the frozen encoder exactly.

In [ ]:
frozen_embeddings = np.load(FROZEN_EMBED_PATH).astype(np.float32)
frozen_embeddings /= np.clip(
    np.linalg.norm(frozen_embeddings, axis=1, keepdims=True),
    1e-12,
    None,
)

finetuned_embeddings /= np.clip(
    np.linalg.norm(finetuned_embeddings, axis=1, keepdims=True),
    1e-12,
    None,
)

same_item_cosine = np.sum(
    frozen_embeddings * finetuned_embeddings,
    axis=1,
)

print('Mean frozen-vs-finetuned same-item cosine  :', float(same_item_cosine.mean()))
print('Median frozen-vs-finetuned same-item cosine:', float(np.median(same_item_cosine)))
print('Min same-item cosine                       :', float(same_item_cosine.min()))

# Part B: Controlled downstream comparison

Everything below uses the exact accepted M4 retrieval/reranking design. The only thing that changes is the metadata embedding matrix.

## 10. Load BPR and reconstruct exact project mappings

In [ ]:
state_dict = torch.load(BPR_CHECKPOINT, map_location='cpu')
bpr_user_embeddings = (
    state_dict['user_embedding.weight'].cpu().numpy().astype(np.float32)
)
bpr_item_embeddings = (
    state_dict['item_embedding.weight'].cpu().numpy().astype(np.float32)
)

all_users = train['user_id'].unique()
all_items = pd.concat([
    train['parent_asin'],
    valid['parent_asin'],
    test['parent_asin'],
]).unique()

user_to_idx = {user: idx for idx, user in enumerate(all_users)}
item_to_idx = {item: idx for idx, item in enumerate(all_items)}

NUM_USERS = len(all_users)
NUM_ITEMS = len(all_items)

assert NUM_USERS == bpr_user_embeddings.shape[0]
assert NUM_ITEMS == bpr_item_embeddings.shape[0]

train_indexed = train[['user_id', 'parent_asin', 'timestamp']].copy()
train_indexed['user_idx'] = train_indexed['user_id'].map(user_to_idx).astype(np.int32)
train_indexed['item_idx'] = train_indexed['parent_asin'].map(item_to_idx).astype(np.int32)

user_seen = (
    train_indexed.groupby('user_idx')['item_idx'].apply(set).to_dict()
)

bpr_catalog_idx = np.array(
    sorted({item_to_idx[item] for item in train['parent_asin'].unique()}),
    dtype=np.int32,
)

global_to_bpr_pos = np.full(NUM_ITEMS, -1, dtype=np.int32)
global_to_bpr_pos[bpr_catalog_idx] = np.arange(len(bpr_catalog_idx), dtype=np.int32)

bpr_trained_mask = np.zeros(NUM_ITEMS, dtype=bool)
bpr_trained_mask[bpr_catalog_idx] = True
bpr_catalog_embeddings = bpr_item_embeddings[bpr_catalog_idx]

valid_user_idx = np.array(
    [user_to_idx[user] for user in valid['user_id'].to_numpy()],
    dtype=np.int32,
)
valid_target_idx = np.array(
    [item_to_idx[item] for item in valid['parent_asin'].to_numpy()],
    dtype=np.int32,
)
NUM_VALID = len(valid_user_idx)

print('Users:', NUM_USERS)
print('Items:', NUM_ITEMS)
print('BPR retrieval catalog:', len(bpr_catalog_idx))
print('Validation users:', NUM_VALID)

## 11. Shared evaluation helpers

In [ ]:
def l2_normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    ok = norms[:, 0] > 1e-12
    matrix[ok] /= norms[ok]
    return matrix, ok


def topk_sorted(scores, k):
    local = np.argpartition(scores, -k, axis=1)[:, -k:]
    top_scores = np.take_along_axis(scores, local, axis=1)
    order = np.argsort(top_scores, axis=1)[:, ::-1]
    local = np.take_along_axis(local, order, axis=1)
    top_scores = np.take_along_axis(top_scores, order, axis=1)
    return local, top_scores


def safe_zscore_1d(values):
    values = np.asarray(values, dtype=np.float32)
    finite = np.isfinite(values)
    out = np.full(values.shape, np.nan, dtype=np.float32)

    if finite.sum() == 0:
        return out
    if finite.sum() == 1:
        out[finite] = 0.0
        return out

    mean = values[finite].mean()
    std = values[finite].std()
    if std < 1e-8:
        out[finite] = 0.0
    else:
        out[finite] = (values[finite] - mean) / std
    return out


def metrics_from_top10(top10, target_idx):
    match = np.where(top10 == int(target_idx))[0]
    if len(match) == 0:
        return 0, 0.0, 0.0
    rank = int(match[0]) + 1
    return 1, 1.0 / rank, 1.0 / np.log2(rank + 1)

## 12. Evaluate one metadata embedding matrix

This function reproduces the accepted M4 logic exactly:

`BPR top-500 + metadata top-500 -> union -> 50/50 z-score reranking -> top-10`

In [ ]:
def evaluate_metadata_variant(metadata_embeddings_variant, label):
    metadata_embeddings_variant = np.asarray(
        metadata_embeddings_variant,
        dtype=np.float32,
    ).copy()
    metadata_embeddings_variant /= np.clip(
        np.linalg.norm(metadata_embeddings_variant, axis=1, keepdims=True),
        1e-12,
        None,
    )

    asin_to_meta_idx = {
        asin: idx for idx, asin in enumerate(metadata_index['parent_asin'])
    }
    global_to_meta_idx = np.array(
        [asin_to_meta_idx.get(asin, -1) for asin in all_items],
        dtype=np.int32,
    )
    assert (global_to_meta_idx >= 0).all()

    global_metadata_embeddings = metadata_embeddings_variant[global_to_meta_idx]
    metadata_catalog_embeddings = global_metadata_embeddings

    indexed = train_indexed.copy()
    indexed['meta_idx'] = indexed['parent_asin'].map(asin_to_meta_idx).astype(np.int32)

    # Long-term metadata profile.
    all_history_matrix = csr_matrix(
        (
            np.ones(len(indexed), dtype=np.float32),
            (indexed['user_idx'], indexed['meta_idx']),
        ),
        shape=(NUM_USERS, len(metadata_index)),
        dtype=np.float32,
    )
    all_history_profiles = np.asarray(
        all_history_matrix @ metadata_embeddings_variant,
        dtype=np.float32,
    )
    all_history_profiles, _ = l2_normalize_rows(all_history_profiles)

    # Recent-2 metadata profile.
    chronological = indexed.sort_values(
        ['user_idx', 'timestamp'],
        kind='mergesort',
    )
    recent_rows = (
        chronological
        .groupby('user_idx', sort=False, group_keys=False)
        .tail(RECENT_N)
    )
    recent_matrix = csr_matrix(
        (
            np.ones(len(recent_rows), dtype=np.float32),
            (recent_rows['user_idx'], recent_rows['meta_idx']),
        ),
        shape=(NUM_USERS, len(metadata_index)),
        dtype=np.float32,
    )
    recent_profiles = np.asarray(
        recent_matrix @ metadata_embeddings_variant,
        dtype=np.float32,
    )
    recent_profiles, _ = l2_normalize_rows(recent_profiles)

    bpr_hits_500 = 0
    meta_hits_500 = 0
    union_hits = 0

    bpr_hr10 = 0
    bpr_mrr10 = 0.0
    bpr_ndcg10 = 0.0

    meta_hr10 = 0
    meta_mrr10 = 0.0
    meta_ndcg10 = 0.0

    hybrid_hr10 = 0
    hybrid_mrr10 = 0.0
    hybrid_ndcg10 = 0.0

    for start in tqdm(
        range(0, NUM_VALID, EVAL_BATCH_SIZE),
        desc=f'Evaluate {label}',
    ):
        end = min(start + EVAL_BATCH_SIZE, NUM_VALID)
        batch_users = valid_user_idx[start:end]
        batch_targets = valid_target_idx[start:end]

        bpr_scores = (
            bpr_user_embeddings[batch_users]
            @ bpr_catalog_embeddings.T
        )
        all_meta_scores = (
            all_history_profiles[batch_users]
            @ metadata_catalog_embeddings.T
        )
        recent_meta_scores = (
            recent_profiles[batch_users]
            @ metadata_catalog_embeddings.T
        )

        for row, user_idx in enumerate(batch_users):
            seen = user_seen.get(int(user_idx), set())
            if not seen:
                continue

            seen_global = np.fromiter(seen, dtype=np.int32)

            bpr_pos = global_to_bpr_pos[seen_global]
            bpr_pos = bpr_pos[bpr_pos >= 0]
            if len(bpr_pos):
                bpr_scores[row, bpr_pos] = -np.inf

            all_meta_scores[row, seen_global] = -np.inf
            recent_meta_scores[row, seen_global] = -np.inf

        meta_scores = (
            (1.0 - RHO) * all_meta_scores
            + RHO * recent_meta_scores
        )

        bpr_local, _ = topk_sorted(bpr_scores, CANDIDATE_K)
        bpr_candidates = bpr_catalog_idx[bpr_local]

        meta_local, _ = topk_sorted(meta_scores, CANDIDATE_K)
        meta_candidates = meta_local.astype(np.int32)

        for r, target_idx in enumerate(batch_targets):
            bpr_list = bpr_candidates[r]
            meta_list = meta_candidates[r]

            bpr_hit500 = bool(np.any(bpr_list == int(target_idx)))
            meta_hit500 = bool(np.any(meta_list == int(target_idx)))

            bpr_hits_500 += int(bpr_hit500)
            meta_hits_500 += int(meta_hit500)
            union_hits += int(bpr_hit500 or meta_hit500)

            h, rr, dcg = metrics_from_top10(bpr_list[:10], target_idx)
            bpr_hr10 += h
            bpr_mrr10 += rr
            bpr_ndcg10 += dcg

            h, rr, dcg = metrics_from_top10(meta_list[:10], target_idx)
            meta_hr10 += h
            meta_mrr10 += rr
            meta_ndcg10 += dcg

            union = np.unique(np.concatenate([bpr_list, meta_list])).astype(np.int32)

            bpr_union_scores = np.full(len(union), np.nan, dtype=np.float32)
            trained = bpr_trained_mask[union]
            if trained.any():
                positions = global_to_bpr_pos[union[trained]]
                bpr_union_scores[trained] = bpr_scores[r, positions]

            meta_union_scores = meta_scores[r, union].astype(np.float32)

            bpr_z = safe_zscore_1d(bpr_union_scores)
            meta_z = safe_zscore_1d(meta_union_scores)

            finite_bpr = np.isfinite(bpr_z)
            if finite_bpr.any():
                bpr_fill = float(bpr_z[finite_bpr].min()) - 1.0
            else:
                bpr_fill = 0.0
            bpr_z = np.where(finite_bpr, bpr_z, bpr_fill)
            meta_z = np.nan_to_num(meta_z, nan=0.0)

            final_scores = ALPHA_BPR * bpr_z + ALPHA_META * meta_z
            top10_local = np.argsort(final_scores)[-10:][::-1]
            final_top10 = union[top10_local]

            h, rr, dcg = metrics_from_top10(final_top10, target_idx)
            hybrid_hr10 += h
            hybrid_mrr10 += rr
            hybrid_ndcg10 += dcg

    return {
        'Variant': label,
        'BPR Recall@500': bpr_hits_500 / NUM_VALID,
        'Metadata Recall@500': meta_hits_500 / NUM_VALID,
        'Union Recall@500': union_hits / NUM_VALID,
        'BPR HR@10': bpr_hr10 / NUM_VALID,
        'BPR MRR@10': bpr_mrr10 / NUM_VALID,
        'BPR NDCG@10': bpr_ndcg10 / NUM_VALID,
        'Metadata HR@10': meta_hr10 / NUM_VALID,
        'Metadata MRR@10': meta_mrr10 / NUM_VALID,
        'Metadata NDCG@10': meta_ndcg10 / NUM_VALID,
        'Hybrid HR@10': hybrid_hr10 / NUM_VALID,
        'Hybrid MRR@10': hybrid_mrr10 / NUM_VALID,
        'Hybrid NDCG@10': hybrid_ndcg10 / NUM_VALID,
    }

## 13. Frozen MiniLM baseline reproduction

In [ ]:
frozen_result = evaluate_metadata_variant(
    frozen_embeddings,
    label='Frozen MiniLM',
)

pd.DataFrame([frozen_result])

### Sanity check against the accepted M4 result

If this fails, do not interpret the fine-tuning comparison. The frozen branch must reproduce the accepted M4 validation metrics first.

In [ ]:
EXPECTED_FROZEN_UNION_RECALL = 0.500338
EXPECTED_FROZEN_HR10 = 0.066250
EXPECTED_FROZEN_NDCG10 = 0.035109

print('Union Recall difference:', frozen_result['Union Recall@500'] - EXPECTED_FROZEN_UNION_RECALL)
print('HR@10 difference       :', frozen_result['Hybrid HR@10'] - EXPECTED_FROZEN_HR10)
print('NDCG@10 difference     :', frozen_result['Hybrid NDCG@10'] - EXPECTED_FROZEN_NDCG10)

assert abs(frozen_result['Union Recall@500'] - EXPECTED_FROZEN_UNION_RECALL) < 5e-6
assert abs(frozen_result['Hybrid HR@10'] - EXPECTED_FROZEN_HR10) < 5e-6
assert abs(frozen_result['Hybrid NDCG@10'] - EXPECTED_FROZEN_NDCG10) < 5e-6

print('Frozen MiniLM pipeline reproduced successfully.')

## 14. Fine-tuned MiniLM evaluation

In [ ]:
finetuned_result = evaluate_metadata_variant(
    finetuned_embeddings,
    label='Fine-tuned MiniLM',
)

comparison = pd.DataFrame([frozen_result, finetuned_result])[[
    'Variant',
    'Metadata Recall@500',
    'Union Recall@500',
    'Metadata HR@10',
    'Metadata NDCG@10',
    'Hybrid HR@10',
    'Hybrid MRR@10',
    'Hybrid NDCG@10',
]]

comparison

## 15. Decision table

In [ ]:
metric_pairs = [
    ('Metadata Recall@500', 'Metadata Recall@500'),
    ('Union Recall@500', 'Union Recall@500'),
    ('Hybrid HR@10', 'Hybrid HR@10'),
    ('Hybrid MRR@10', 'Hybrid MRR@10'),
    ('Hybrid NDCG@10', 'Hybrid NDCG@10'),
]

rows = []
for label, key in metric_pairs:
    old = float(frozen_result[key])
    new = float(finetuned_result[key])
    rows.append({
        'Metric': label,
        'Frozen': old,
        'Fine-tuned': new,
        'Absolute change': new - old,
        'Relative change (%)': ((new / old) - 1.0) * 100.0 if old != 0 else np.nan,
    })

decision_table = pd.DataFrame(rows)
decision_table

In [ ]:
if finetuned_result['Hybrid NDCG@10'] > frozen_result['Hybrid NDCG@10']:
    print('KEEP candidate: fine-tuned MiniLM improved the primary metric NDCG@10.')
    print('Do not overwrite the frozen embeddings yet; update M4 only after reviewing HR/recall too.')
else:
    print('REJECT candidate: fine-tuned MiniLM did not improve NDCG@10.')
    print('Keep the frozen MiniLM architecture in M4.')

## Interpretation

This notebook deliberately does not retune `rho`, `alpha`, or candidate size for the fine-tuned encoder. That keeps the first comparison controlled: the **only changed component is MiniLM itself**.

If fine-tuning clearly improves full-VALID NDCG@10 without materially damaging HR@10 or candidate recall, then the next step is to update M4 to load `metadata_minilm_finetuned_embeddings.npy` and optionally perform a very small local recheck of `rho/alpha`.

If it does not improve, keep the frozen MiniLM result and stop this branch. No TEST evaluation should be performed from this notebook.